In [1]:
import json
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib import rcParams as rc

rc["font.family"] = "Times New Roman"
rc["font.size"] = 14
rc["figure.figsize"] = (6, 4)
rc["axes.grid"] = True

In [2]:
# Load the configuration file
conf = json.load(open("../data/atem.json"))
times = np.asarray(conf['channels']) * 1e-6
n_turns = conf['n_turns']

In [3]:
area:str = "NE"
path:str = f"../data/11-024_Alberta_{area}.csv"
dheader:list = [f"zoff30[{i}]" for i in range(30)]
picker:list = ["Line", "bheight", "TranPeak", "x_wgs84", "y_wgs84", "flight", 'pwrline'] + dheader # power line monitor

In [ ]:
dobs = pd.read_csv(path)[picker]

In [ ]:
xy = dobs[["x_wgs84", "y_wgs84"]].to_numpy()
normalizer = (-1e-9)/ (dobs["TranPeak"].values * n_turns).reshape(-1, 1)
dobs[[f"zoff30[{i}]" for i in range(30)]] = dobs[[f"zoff30[{i}]" for i in range(30)]] * normalizer
floors = 5 * normalizer 

In [ ]:
line_no = list(dobs["Line"].unique())

In [ ]:
print(line_no)

In [ ]:
istart:int = 19
iend:int = 30
index = dobs["Line"] == line_no[istart]
print(f"{line_no[istart]=}")

In [ ]:
plt.scatter(xy[:, 0], xy[:, 1], s=1)
plt.scatter(xy[index, 0], xy[index, 1], s=1)
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(f"Line {line_no[istart]}")

In [ ]:
dobs.fillna(1e-20)

In [ ]:
dx = 25.
values = []
values_std = []
soundings = []

for i_line, line in enumerate(line_no[istart:iend]):
    df_line = dobs[dobs['Line']==line]

    # Calculate distance along the "Line"
    xy = df_line[["x_wgs84", "y_wgs84"]].to_numpy()
    distance = np.sqrt(((xy-xy[0,:])**2).sum(axis=1))
    max_distance = distance.max()

    # Determine the no. of soundings per bin.
    if max_distance % dx ==0:
        n_sounding = int(max_distance / dx)
    else:
        n_sounding = int(np.round(max_distance / dx) + 1)

    # Create bins and assign each sounding to a bin
    bins = np.arange(n_sounding) * dx
    df_line.insert(0, 'distance', distance)
    # Bin distances
    df_line['bin'] = pd.cut(df_line['distance'], bins=bins)
    # Compute statistics per bin
    binned = (
        df_line.groupby('bin', observed=False)
            [['distance', 'x_wgs84', 'y_wgs84'] + picker[1:]]
            .mean()
    )
    binned.insert(0, 'Line', line)
    binned_std = (
        df_line.groupby('bin', observed=False)
            [['bheight'] + dheader]
            .std()
    )
    values.append(binned.values)
    values_std.append(binned_std.values)
    soundings.append(n_sounding)

df_data_binned = pd.DataFrame(data=np.vstack(values), columns=['Line', 'distance', 'x_wgs84', 'y_wgs84'] + picker[1:])
df_data_std_binned = pd.DataFrame(data=np.vstack(values_std), columns=['bheight'] + dheader)

In [ ]:
print(f"{soundings=}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from ipywidgets import widgets, interact

In [ ]:
def plot_data(line, x):
    fig, axs = plt.subplots(2,1, figsize=(20, 10), constrained_layout=True)
    ax0, ax1 = axs

    x_line = x[x["Line"]==line]

    # (ax0) Plot the binned locations
    ax0.plot(x["x_wgs84"], x["y_wgs84"], '.')
    ax0.plot(x_line["x_wgs84"], x_line["y_wgs84"], '.')
    ax0.plot(x_line["x_wgs84"].values[0], x_line["y_wgs84"].values[0], 'o')
    ax0.set_aspect(0.2)

    # Normalize the altitude and power line monitor data.
    scaler = MinMaxScaler(feature_range=(x_line['bheight'].min(), x_line['bheight'].max())) # Min-Max scaler for pwrline to be in the same range as bheight
    plm_norm = scaler.fit_transform(x_line['pwrline'].values.reshape([-1,1]))

    # (ax1) Plot the normalized data
    ax2 = ax1.twinx()
    ax1.semilogy(x_line['distance'], np.abs(x_line[dheader]), color='k', lw=2)
    ax2.plot(x_line['distance'], plm_norm, '-', color='red', label='PLM', lw=1.5)
    ax2.plot(x_line['distance'], x_line['bheight'], '-', label='Flight height', lw=1.5, color='magenta')
    ax2.legend()

In [ ]:
interact(plot_data, line=widgets.Select(options=line_no[istart:iend]), x=widgets.fixed(df_data_binned))

- standard deviation
$$
\sigma=\frac{1}{N}\sqrt{\sum_{i=1^N}{(x_i-\bar{x})^2}}
$$
- field normalized standard deviation (root-mean-squared relative error)
$$
\frac{\sigma}{d}=\frac{1}{N}\sqrt{\sum_{i=1^N}{\frac{(x_i-\bar{x})^2}{x_i^2}}}
$$

In [ ]:
# Extract timeseries data
data = df_data_binned[dheader].values.astype(float)
# Extract field normalized standard deviation (root mean squared relative error)
data_rerr = (df_data_std_binned[dheader].values / np.abs(df_data_binned[dheader].values)).astype(float)

In [ ]:
# Cut-off for bad data (relative error > 5%)
channel_id = np.tile(np.arange(data.shape[1]), (data.shape[0], 1))
cut_off = (data_rerr>0.05) * (channel_id>=0)

In [ ]:
data[cut_off] = np.nan
data_rerr[cut_off] = np.nan

In [ ]:
# Plot histogram of relative errors below the cut-off
hi = plt.hist(data_rerr[~cut_off], bins = np.linspace(0,0.05, 100))

In [ ]:
filtered_data = pd.DataFrame(
    data = np.hstack(
        (df_data_binned.iloc[:,:10].values, data)
    ),
    columns=df_data_binned.columns.tolist()
)

In [ ]:
interact(plot_data, line=widgets.Select(options=line_no[istart:iend]), x=widgets.fixed(filtered_data))

# Todo: 
- Data bining (required review) 
- Figure out the meaning of pwrline (power line management) range.
- Inversion